1 — Mount Drive and Setup

In [ ]:
# ============================================================
# NOTEBOOK 5 — YOLOv5 DETECTION FEATURE EXTRACTION
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


from pathlib import Path
import pandas as pd
import numpy as np
import torch
from tqdm.auto import tqdm
import json
import gc
import os


# ============================================================
# PROJECT PATHS
# ============================================================

PROJECT_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning"
)

IMAGE_MANIFEST_PATH = (
    PROJECT_ROOT
    / "Dataset"
    / "Processed"
    / "Images"
    / "image_manifest.csv"
)


# ============================================================
# YOLOV5 OUTPUT DIRECTORY
# ============================================================

YOLOV5_OUTPUT_DIR = (
    PROJECT_ROOT
    / "Outputs"
    / "YOLOv5_Detection"
)

YOLOV5_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("NOTEBOOK 5 — YOLOv5 DETECTION")
print("=" * 70)

print("\nProject:")
print(PROJECT_ROOT)

print("\nInput manifest:")
print(IMAGE_MANIFEST_PATH)

print("\nOutput directory:")
print(YOLOV5_OUTPUT_DIR)


assert PROJECT_ROOT.exists(), (
    "Project folder not found."
)

assert IMAGE_MANIFEST_PATH.exists(), (
    "image_manifest.csv not found. "
    "Notebook 4 must be completed first."
)


print("\n✓ Drive mounted")
print("✓ Notebook 4 output found")
print("✓ YOLOv5 output directory ready")

Mounted at /content/drive
NOTEBOOK 5 — YOLOv5 DETECTION

Project:
/content/drive/MyDrive/Image_captioning

Input manifest:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Images/image_manifest.csv

Output directory:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection

✓ Drive mounted
✓ Notebook 4 output found
✓ YOLOv5 output directory ready


2 — Check GPU

In [ ]:
# ============================================================
# GPU VERIFICATION
# ============================================================

print("=" * 70)
print("GPU VERIFICATION")
print("=" * 70)


print("\nPyTorch version:")
print(torch.__version__)


print("\nCUDA available:")
print(torch.cuda.is_available())


if torch.cuda.is_available():

    print("\nGPU:")
    print(
        torch.cuda.get_device_name(0)
    )

    DEVICE = "cuda"

else:

    DEVICE = "cpu"


print("\nSelected device:")
print(DEVICE)


assert torch.cuda.is_available(), (
    "\nGPU IS NOT ENABLED.\n"
    "Stop here.\n"
    "In Colab go to:\n"
    "Runtime → Change runtime type → GPU\n"
    "Then reconnect and rerun this notebook."
)

`
print("\n✓ GPU READY")

GPU VERIFICATION

PyTorch version:
2.11.0+cu128

CUDA available:
True

GPU:
Tesla T4

Selected device:
cuda

✓ GPU READY


3 — Load Verified Image Manifest

In [ ]:
# ============================================================
# LOAD VERIFIED IMAGE MANIFEST
# ============================================================

images_df = pd.read_csv(
    IMAGE_MANIFEST_PATH
)


print("=" * 70)
print("IMAGE MANIFEST LOADED")
print("=" * 70)


print(
    "\nShape:",
    images_df.shape
)


print(
    "\nColumns:"
)

print(
    list(images_df.columns)
)


print(
    "\nTotal images:"
)

print(
    len(images_df)
)


print(
    "\nUnique images:"
)

print(
    images_df["image_id"]
    .nunique()
)


print(
    "\nDataset distribution:"
)

print(
    images_df["dataset"]
    .value_counts()
)


print(
    "\nMissing paths:"
)

print(
    images_df["image_path"]
    .isna()
    .sum()
)


assert len(images_df) == 39874

assert (
    images_df["image_id"]
    .nunique()
    == 39874
)

assert (
    images_df["image_path"]
    .isna()
    .sum()
    == 0
)


print("\n✓ VERIFIED IMAGE MANIFEST READY")

IMAGE MANIFEST LOADED

Shape: (39874, 4)

Columns:
['image_id', 'dataset', 'image_name', 'image_path']

Total images:
39874

Unique images:
39874

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

Missing paths:
0

✓ VERIFIED IMAGE MANIFEST READY


4 — Verify a Few Image Paths

In [ ]:
# ============================================================
# QUICK PATH VERIFICATION
# ============================================================

print("=" * 70)
print("QUICK IMAGE PATH VERIFICATION")
print("=" * 70)


sample_df = images_df.sample(
    n=10,
    random_state=42
)


missing = []


for _, row in sample_df.iterrows():

    image_path = Path(
        row["image_path"]
    )

    if not image_path.exists():

        missing.append(
            row["image_id"]
        )


print(
    "\nSample images checked:",
    len(sample_df)
)

print(
    "Missing sample files:",
    len(missing)
)


if missing:

    print("\nMissing examples:")

    for image_id in missing:
        print(image_id)


assert len(missing) == 0


print("\n✓ IMAGE PATHS READY")

QUICK IMAGE PATH VERIFICATION

Sample images checked: 10
Missing sample files: 0

✓ IMAGE PATHS READY


5 — Install Dependencies and Load YOLOv5

In [ ]:
# ============================================================
# CELL 5 — INSTALL DEPENDENCIES AND LOAD YOLOv5
# ============================================================

print("=" * 70)
print("INSTALLING YOLOv5 DEPENDENCIES")
print("=" * 70)

!pip install -q ultralytics

print("\n✓ Ultralytics package installed")


# ============================================================
# LOAD YOLOv5
# ============================================================

import torch

print("\n" + "=" * 70)
print("LOADING YOLOv5")
print("=" * 70)


yolo_model = torch.hub.load(
    "ultralytics/yolov5",
    "yolov5s",
    pretrained=True,
    trust_repo=True
)


# ------------------------------------------------------------
# MOVE MODEL TO GPU
# ------------------------------------------------------------

yolo_model.to(DEVICE)

yolo_model.eval()


# ------------------------------------------------------------
# DETECTION SETTINGS
# ------------------------------------------------------------

CONFIDENCE_THRESHOLD = 0.25
IOU_THRESHOLD = 0.45
IMAGE_SIZE = 640


yolo_model.conf = CONFIDENCE_THRESHOLD

yolo_model.iou = IOU_THRESHOLD

yolo_model.classes = None

yolo_model.max_det = 100


print("\nModel:")
print("YOLOv5s")

print("\nDevice:")
print(DEVICE)

print("\nConfidence threshold:")
print(CONFIDENCE_THRESHOLD)

print("\nIoU threshold:")
print(IOU_THRESHOLD)

print("\nImage size:")
print(IMAGE_SIZE)


print("\n✓ YOLOv5 LOADED SUCCESSFULLY")

INSTALLING YOLOv5 DEPENDENCIES
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.0/46.0 kB 3.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 77.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.3 MB/s eta 0:00:00

✓ Ultralytics package installed

LOADING YOLOv5
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.


Using cache found in /root/.cache/torch/hub/ultralytics_yolov5_master


YOLOv5 🚀 2026-9-2 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)



100%|██████████| 14.1M/14.1M [00:00<00:00, 403MB/s]

Fusing layers... 
YOLOv5s summary: 124 layers, 7,225,885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 

Model:
YOLOv5s

Device:
cuda

Confidence threshold:
0.25

IoU threshold:
0.45

Image size:
640

✓ YOLOv5 LOADED SUCCESSFULLY


6 — Test YOLOv5 on One Image

In [ ]:
# ============================================================
# SINGLE IMAGE YOLOv5 TEST
# ============================================================

print("=" * 70)
print("YOLOv5 SINGLE IMAGE TEST")
print("=" * 70)


test_row = images_df.iloc[0]

test_image_path = (
    test_row["image_path"]
)


print(
    "\nImage ID:"
)

print(
    test_row["image_id"]
)


print(
    "\nImage path:"
)

print(
    test_image_path
)


# ------------------------------------------------------------
# RUN DETECTION
# ------------------------------------------------------------

with torch.no_grad():

    test_results = yolo_model(
        test_image_path,
        size=IMAGE_SIZE
    )


test_predictions = (
    test_results
    .pandas()
    .xyxy[0]
)


print(
    "\nDetections found:",
    len(test_predictions)
)


if len(test_predictions) > 0:

    print("\nDetections:")

    display(
        test_predictions[
            [
                "name",
                "confidence",
                "xmin",
                "ymin",
                "xmax",
                "ymax"
            ]
        ]
    )

else:

    print(
        "\nNo objects detected."
    )


print("\n✓ SINGLE IMAGE TEST COMPLETE")

YOLOv5 SINGLE IMAGE TEST

Image ID:
Flickr8K/1000268201_693b08cb0e.jpg

Image path:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K/Images/1000268201_693b08cb0e.jpg

Detections found: 2

Detections:


,name,confidence,xmin,ymin,xmax,ymax
0,person,0.848487,5.304271,151.377350,106.736732,338.762726
1,vase,0.452599,284.475403,234.829361,298.211060,277.459564



✓ SINGLE IMAGE TEST COMPLETE


7 — Define YOLOv5 Batch Detection Function

In [ ]:
# ============================================================
# CELL 7 — YOLOv5 BATCH DETECTION FUNCTION
# ============================================================

from pathlib import Path
import pandas as pd
import torch
import gc


def run_yolov5_batch(image_paths):
    """
    Runs YOLOv5 detection on a batch of image paths.

    Returns one detection record for each detected object.
    """

    records = []

    with torch.no_grad():

        results = yolo_model(
            image_paths,
            size=IMAGE_SIZE
        )


    predictions = (
        results
        .pandas()
        .xyxy
    )


    # --------------------------------------------------------
    # PROCESS EACH IMAGE'S DETECTIONS
    # --------------------------------------------------------

    for image_path, prediction_df in zip(
        image_paths,
        predictions
    ):

        image_path = str(
            image_path
        )


        # Find the original manifest row
        image_id = path_to_id[
            image_path
        ]


        # ----------------------------------------------------
        # CASE: NO OBJECT DETECTED
        # ----------------------------------------------------

        if len(prediction_df) == 0:

            records.append(
                {
                    "image_id": image_id,
                    "image_path": image_path,
                    "object_name": None,
                    "confidence": None,
                    "xmin": None,
                    "ymin": None,
                    "xmax": None,
                    "ymax": None,
                    "objects_detected": 0
                }
            )

            continue


        # ----------------------------------------------------
        # SAVE EACH DETECTED OBJECT
        # ----------------------------------------------------

        for _, detection in prediction_df.iterrows():

            records.append(
                {
                    "image_id": image_id,
                    "image_path": image_path,
                    "object_name": detection["name"],
                    "confidence": float(
                        detection["confidence"]
                    ),
                    "xmin": float(
                        detection["xmin"]
                    ),
                    "ymin": float(
                        detection["ymin"]
                    ),
                    "xmax": float(
                        detection["xmax"]
                    ),
                    "ymax": float(
                        detection["ymax"]
                    ),
                    "objects_detected": len(
                        prediction_df
                    )
                }
            )


    # --------------------------------------------------------
    # MEMORY CLEANUP
    # --------------------------------------------------------

    del results
    del predictions

    gc.collect()

    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return records


print(
    "✓ YOLOv5 batch detection function ready"
)

✓ YOLOv5 batch detection function ready


8 — Build Image Path Lookup

In [ ]:
# ============================================================
# CELL 8 — BUILD FAST IMAGE LOOKUP
# ============================================================

path_to_id = dict(
    zip(
        images_df["image_path"].astype(str),
        images_df["image_id"]
    )
)


all_image_paths = (
    images_df["image_path"]
    .astype(str)
    .tolist()
)


print("=" * 70)
print("IMAGE LOOKUP READY")
print("=" * 70)

print(
    "\nTotal image paths:",
    len(all_image_paths)
)

print(
    "Lookup entries:",
    len(path_to_id)
)


assert (
    len(all_image_paths)
    == 39874
)

assert (
    len(path_to_id)
    == 39874
)


print(
    "\n✓ READY FOR BATCH DETECTION"
)

IMAGE LOOKUP READY

Total image paths: 39874
Lookup entries: 39874

✓ READY FOR BATCH DETECTION


9 — Small Batch Test

In [ ]:
# ============================================================
# CELL 9 — YOLOv5 SMALL BATCH TEST
# ============================================================

TEST_BATCH_SIZE = 8


test_image_paths = (
    all_image_paths[
        :TEST_BATCH_SIZE
    ]
)


print("=" * 70)
print("YOLOv5 BATCH TEST")
print("=" * 70)

print(
    "\nImages in test batch:",
    len(test_image_paths)
)


test_records = run_yolov5_batch(
    test_image_paths
)


test_detections_df = pd.DataFrame(
    test_records
)


print(
    "\nDetection records:",
    len(test_detections_df)
)


print(
    "\nUnique images processed:",
    test_detections_df[
        "image_id"
    ].nunique()
)


print(
    "\nSample detections:"
)

display(
    test_detections_df.head(20)
)


assert (
    test_detections_df[
        "image_id"
    ].nunique()
    == TEST_BATCH_SIZE
)


print(
    "\n✓ YOLOv5 BATCH TEST SUCCESSFUL"
)

YOLOv5 BATCH TEST

Images in test batch: 8

Detection records: 16

Unique images processed: 8

Sample detections:


,image_id,image_path,object_name,confidence,xmin,ymin,xmax,ymax,objects_detected
0,Flickr8K/1000268201_693b08cb0e.jpg,/content/drive/MyDrive/Image_captioning/Datase...,person,0.895094,5.507493,150.834244,107.664749,339.458649,3
1,Flickr8K/1000268201_693b08cb0e.jpg,/content/drive/MyDrive/Image_captioning/Datase...,vase,0.430035,284.895966,234.392075,298.524323,277.204376,3
2,Flickr8K/1000268201_693b08cb0e.jpg,/content/drive/MyDrive/Image_captioning/Datase...,potted plant,0.292970,265.469421,122.163155,373.437592,261.090881,3
3,Flickr8K/1001773457_577c3a7d70.jpg,/content/drive/MyDrive/Image_captioning/Datase...,dog,0.928686,0.170046,1.851481,189.406036,152.445007,2
4,Flickr8K/1001773457_577c3a7d70.jpg,/content/drive/MyDrive/Image_captioning/Datase...,dog,0.759649,234.788422,101.904472,418.347443,372.007080,2
5,Flickr8K/1002674143_1b742ab4b8.jpg,/content/drive/MyDrive/Image_captioning/Datase...,person,0.673216,108.957001,234.397583,193.316406,361.793427,2
6,Flickr8K/1002674143_1b742ab4b8.jpg,/content/drive/MyDrive/Image_captioning/Datase...,kite,0.345895,38.539459,2.107084,483.094299,364.579498,2
7,Flickr8K/1003163366_44323f5815.jpg,/content/drive/MyDrive/Image_captioning/Datase...,dog,0.845813,98.831879,258.041748,199.463379,389.719269,2
8,Flickr8K/1003163366_44323f5815.jpg,/content/drive/MyDrive/Image_captioning/Datase...,person,0.591479,107.682373,81.798759,378.460846,250.944443,2
9,Flickr8K/1007129816_e794419615.jpg,/content/drive/MyDrive/Image_captioning/Datase...,person,0.538328,17.701101,69.928139,467.046753,461.000000,3



✓ YOLOv5 BATCH TEST SUCCESSFUL


10 — Create Checkpoint and Final Output Directories

In [ ]:
# ============================================================
# CELL 10 — YOLOv5 OUTPUT DIRECTORIES
# ============================================================

CHECKPOINT_DIR = (
    YOLOV5_OUTPUT_DIR
    / "Checkpoints"
)

FINAL_DIR = (
    YOLOV5_OUTPUT_DIR
    / "Final"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


print("=" * 70)
print("YOLOv5 OUTPUT DIRECTORIES")
print("=" * 70)

print("\nCheckpoint directory:")
print(CHECKPOINT_DIR)

print("\nFinal directory:")
print(FINAL_DIR)

print("\n✓ OUTPUT DIRECTORIES READY")

YOLOv5 OUTPUT DIRECTORIES

Checkpoint directory:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints

Final directory:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Final

✓ OUTPUT DIRECTORIES READY


11 — Configure Full Dataset Processing

In [ ]:
# ============================================================
# CELL 11 — FAST PROCESSING CONFIGURATION
# ============================================================

BATCH_SIZE = 64

CHECKPOINT_EVERY_IMAGES = 512


print("=" * 70)
print("YOLOv5 FAST PROCESSING CONFIGURATION")
print("=" * 70)

print("\nGPU:")
print(torch.cuda.get_device_name(0))

print("\nBatch size:", BATCH_SIZE)

print("\nCheckpoint interval:", CHECKPOINT_EVERY_IMAGES)

print("\nTotal images:", len(all_image_paths))

TOTAL_BATCHES = (
    len(all_image_paths) + BATCH_SIZE - 1
) // BATCH_SIZE

print("\nEstimated batches:", TOTAL_BATCHES)

print("\n✓ FAST CONFIGURATION READY")

YOLOv5 FAST PROCESSING CONFIGURATION

GPU:
Tesla T4

Batch size: 64

Checkpoint interval: 512

Total images: 39874

Estimated batches: 624

✓ FAST CONFIGURATION READY


In [ ]:
# ============================================================
# SPEED TEST — 64 IMAGES
# ============================================================

import time
import torch

speed_test_paths = all_image_paths[:64]

print("=" * 70)
print("YOLOv5 SPEED TEST")
print("=" * 70)

print("\nTesting batch size:", len(speed_test_paths))

torch.cuda.synchronize()

start_time = time.time()

speed_test_records = run_yolov5_batch(
    speed_test_paths
)

torch.cuda.synchronize()

elapsed = time.time() - start_time

print("\nImages processed:", len(speed_test_paths))
print("Detection records:", len(speed_test_records))
print(f"Time taken: {elapsed:.2f} seconds")
print(
    f"Speed: "
    f"{len(speed_test_paths) / elapsed:.2f} images/second"
)

print("\nGPU memory allocated:")
print(
    f"{torch.cuda.memory_allocated() / 1024**3:.2f} GB"
)

print("\n✓ SPEED TEST COMPLETE")

YOLOv5 SPEED TEST

Testing batch size: 64

Images processed: 64
Detection records: 253
Time taken: 1.54 seconds
Speed: 41.54 images/second

GPU memory allocated:
0.04 GB

✓ SPEED TEST COMPLETE


12 — Full YOLOv5 Processing With Automatic

In [ ]:
print(list(CHECKPOINT_DIR.glob("checkpoint_*.csv")))

[]


In [ ]:
# ============================================================
# CELL 12 — FULL YOLOv5 PROCESSING WITH RESUME
# ============================================================
A — Mount Drive + imports + paths
import re
from tqdm.auto import tqdm


print("=" * 70)
print("YOLOv5 FULL DATASET PROCESSING")
print("=" * 70)


# ------------------------------------------------------------
# FIND EXISTING CHECKPOINTS
# ------------------------------------------------------------

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    ),
    key=lambda x: x.name
)


processed_image_ids = set()


if checkpoint_files:

    print(
        "\nExisting checkpoints found:",
        len(checkpoint_files)
    )

    print(
        "Loading processed image IDs..."
    )


    for checkpoint_file in checkpoint_files:

        checkpoint_df = pd.read_csv(
            checkpoint_file
        )

        processed_image_ids.update(
            checkpoint_df[
                "image_id"
            ]
            .dropna()
            .unique()
        )


    print(
        "Images already completed:",
        len(processed_image_ids)
    )

else:

    print(
        "\nNo previous checkpoints found."
    )



# ------------------------------------------------------------
# FIND REMAINING IMAGES
# ------------------------------------------------------------

remaining_image_paths = [
    path
    for path in all_image_paths
    if path_to_id[path]
    not in processed_image_ids
]


print(
    "\nRemaining images:",
    len(remaining_image_paths)
)


print(
    "\nTotal dataset images:",
    len(all_image_paths)
)


if len(remaining_image_paths) == 0:

    print(
        "\n✓ ALL IMAGES ALREADY PROCESSED"
    )

else:

    # --------------------------------------------------------
    # PROCESS REMAINING IMAGES
    # --------------------------------------------------------

    checkpoint_records = []

    images_since_checkpoint = 0

    checkpoint_number = (
        len(checkpoint_files)
        + 1
    )


    for start in tqdm(
        range(
            0,
            len(remaining_image_paths),
            BATCH_SIZE
        ),
        desc="YOLOv5 Detection"
    ):


        batch_paths = (
            remaining_image_paths[
                start:
                start + BATCH_SIZE
            ]
        )


        # ----------------------------------------------------
        # RUN DETECTION
        # ----------------------------------------------------

        batch_records = (
            run_yolov5_batch(
                batch_paths
            )
        )


        checkpoint_records.extend(
            batch_records
        )


        images_since_checkpoint += (
            len(batch_paths)
        )


        # ----------------------------------------------------
        # SAVE CHECKPOINT
        # ----------------------------------------------------

        if (
            images_since_checkpoint
            >= CHECKPOINT_EVERY_IMAGES
        ):

            checkpoint_path = (
                CHECKPOINT_DIR
                / (
                    f"checkpoint_"
                    f"{checkpoint_number:05d}.csv"
                )
            )


            checkpoint_df = (
                pd.DataFrame(
                    checkpoint_records
                )
            )


            checkpoint_df.to_csv(
                checkpoint_path,
                index=False
            )


            print(
                f"\n✓ Checkpoint "
                f"{checkpoint_number} saved"
            )

            print(
                "Images in checkpoint:",
                checkpoint_df[
                    "image_id"
                ]
                .nunique()
            )


            checkpoint_records = []

            images_since_checkpoint = 0

            checkpoint_number += 1


    # --------------------------------------------------------
    # SAVE FINAL PARTIAL CHECKPOINT
    # --------------------------------------------------------

    if checkpoint_records:

        checkpoint_path = (
            CHECKPOINT_DIR
            / (
                f"checkpoint_"
                f"{checkpoint_number:05d}.csv"
            )
        )


        checkpoint_df = (
            pd.DataFrame(
                checkpoint_records
            )
        )


        checkpoint_df.to_csv(
            checkpoint_path,
            index=False
        )


        print(
            f"\n✓ Final checkpoint "
            f"{checkpoint_number} saved"
        )


    print(
        "\n✓ YOLOv5 PROCESSING COMPLETE"
    )

YOLOv5 FULL DATASET PROCESSING

No previous checkpoints found.

Remaining images: 39874

Total dataset images: 39874


YOLOv5 Detection:   0%|          | 0/624 [00:00<?, ?it/s]


✓ Checkpoint 1 saved
Images in checkpoint: 512

✓ Checkpoint 2 saved
Images in checkpoint: 512

✓ Checkpoint 3 saved
Images in checkpoint: 512


KeyboardInterrupt: 

A — Mount Drive + imports + paths

In [ ]:
# ============================================================
# CELL A — RESTORE COLAB ENVIRONMENT
# ============================================================

from google.colab import drive

drive.mount("/content/drive")


import os
import gc
import time
from pathlib import Path

import pandas as pd
import numpy as np
import torch

from tqdm.auto import tqdm


# ------------------------------------------------------------
# PROJECT PATH
# ------------------------------------------------------------

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Image_captioning"
)


# ------------------------------------------------------------
# YOLO OUTPUT PATH
# ------------------------------------------------------------

YOLOV5_OUTPUT_DIR = (
    PROJECT_DIR
    / "Features"
    / "YOLOv5"
)


CHECKPOINT_DIR = (
    YOLOV5_OUTPUT_DIR
    / "Checkpoints"
)


FINAL_DIR = (
    YOLOV5_OUTPUT_DIR
    / "Final"
)


CHECKPOINT_DIR.mkdir(
    parents=True,
    exist_ok=True
)

FINAL_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# IMAGE SIZE
# ------------------------------------------------------------

IMAGE_SIZE = 640


print("=" * 70)
print("ENVIRONMENT RESTORED")
print("=" * 70)

print("\nProject:")
print(PROJECT_DIR)

print("\nCheckpoint directory:")
print(CHECKPOINT_DIR)

print("\nGPU available:")
print(torch.cuda.is_available())

if torch.cuda.is_available():
    print(
        "GPU:",
        torch.cuda.get_device_name(0)
    )

Mounted at /content/drive
ENVIRONMENT RESTORED

Project:
/content/drive/MyDrive/Image_captioning

Checkpoint directory:
/content/drive/MyDrive/Image_captioning/Features/YOLOv5/Checkpoints

GPU available:
True
GPU: Tesla T4


B — Automatically find and load your Image

In [ ]:
# ============================================================
# CELL B — FIND AND LOAD IMAGE MANIFEST
# ============================================================

print("=" * 70)
print("LOCATING IMAGE MANIFEST")
print("=" * 70)


csv_files = list(
    PROJECT_DIR.rglob("*.csv")
)


candidate_files = []


for file_path in csv_files:

    try:

        test_df = pd.read_csv(
            file_path,
            nrows=5
        )


        required_columns = {
            "image_id",
            "image_path"
        }


        if required_columns.issubset(
            test_df.columns
        ):

            candidate_files.append(
                file_path
            )

    except Exception:

        pass


print(
    "\nCandidate image manifest files:"
)


for file_path in candidate_files:

    print(
        file_path
    )


assert len(candidate_files) > 0, (
    "No image manifest containing "
    "'image_id' and 'image_path' was found."
)


# Load the largest valid manifest
best_manifest = None
best_rows = 0


for file_path in candidate_files:

    try:

        df = pd.read_csv(
            file_path
        )

        if len(df) > best_rows:

            best_rows = len(df)
            best_manifest = file_path

    except Exception:

        pass


images_df = pd.read_csv(
    best_manifest
)


print("\n" + "=" * 70)
print("IMAGE MANIFEST LOADED")
print("=" * 70)

print(
    "\nFile:"
)

print(
    best_manifest
)


print(
    "\nShape:",
    images_df.shape
)


print(
    "\nColumns:"
)

print(
    images_df.columns.tolist()
)


print(
    "\nUnique images:",
    images_df["image_id"].nunique()
)


assert (
    images_df["image_id"].nunique()
    == 39874
), (
    "Expected 39,874 unique images."
)


print(
    "\n✓ IMAGE MANIFEST RESTORED SUCCESSFULLY"
)

LOCATING IMAGE MANIFEST

Candidate image manifest files:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints/checkpoint_00001.csv
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints/checkpoint_00002.csv
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints/checkpoint_00003.csv
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Images/image_manifest.csv

IMAGE MANIFEST LOADED

File:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Images/image_manifest.csv

Shape: (39874, 4)

Columns:
['image_id', 'dataset', 'image_name', 'image_path']

Unique images: 39874

✓ IMAGE MANIFEST RESTORED SUCCESSFULLY


CELL C — Rebuild image lookup

In [ ]:
# ============================================================
# CELL C — REBUILD IMAGE LOOKUP
# ============================================================

images_df = (
    images_df
    .drop_duplicates(
        subset=["image_id"]
    )
    .copy()
)


images_df["image_path"] = (
    images_df["image_path"]
    .astype(str)
)


path_to_id = dict(
    zip(
        images_df["image_path"],
        images_df["image_id"]
    )
)


all_image_paths = (
    images_df["image_path"]
    .tolist()
)


print("=" * 70)
print("IMAGE LOOKUP READY")
print("=" * 70)

print(
    "\nTotal images:",
    len(all_image_paths)
)

print(
    "Lookup entries:",
    len(path_to_id)
)


assert len(all_image_paths) == 39874
assert len(path_to_id) == 39874


print(
    "\n✓ READY"
)

IMAGE LOOKUP READY

Total images: 39874
Lookup entries: 39874

✓ READY


CELL D — Install/load YOLOv5 again

In [ ]:
# ============================================================
# CELL D — LOAD YOLOv5
# ============================================================

!pip install -q ultralytics


print("=" * 70)
print("LOADING YOLOv5")
print("=" * 70)


device = (
    "cuda"
    if torch.cuda.is_available()
    else "cpu"
)


yolo_model = torch.hub.load(
    "ultralytics/yolov5",
    "yolov5s",
    pretrained=True,
    trust_repo=True
)


yolo_model.to(
    device
)


yolo_model.eval()


# Detection confidence threshold
yolo_model.conf = 0.25


print(
    "\nDevice:",
    device
)


print(
    "\nModel:",
    "YOLOv5s"
)


print(
    "\n✓ YOLOv5 READY"
)

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.2/46.2 kB 936.1 kB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 66.2/66.2 kB 7.3 MB/s eta 0:00:00
LOADING YOLOv5
Downloading: "https://github.com/ultralytics/yolov5/zipball/master" to /root/.cache/torch/hub/master.zip
Creating new Ultralytics Settings v0.0.8 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart#ultralytics-settings.
YOLOv5 🚀 2026-9-6 Python-3.13.15 torch-2.11.0+cu128 CUDA:0 (Tesla T4, 14913MiB)



100%|██████████| 14.1M/14.1M [00:00<00:00, 121MB/s] 

Fusing layers... 
YOLOv5s summary: 124 layers, 7,225,885 parameters, 0 gradients, 16.4 GFLOPs
Adding AutoShape... 

Device: cuda

Model: YOLOv5s

✓ YOLOv5 READY


CELL E — Restore the batch detection function

In [ ]:
# ============================================================
# CELL E — YOLOv5 BATCH DETECTION FUNCTION
# ============================================================

def run_yolov5_batch(image_paths):

    records = []


    with torch.no_grad():

        results = yolo_model(
            image_paths,
            size=IMAGE_SIZE
        )


    predictions = (
        results
        .pandas()
        .xyxy
    )


    for image_path, prediction_df in zip(
        image_paths,
        predictions
    ):

        image_path = str(
            image_path
        )


        image_id = (
            path_to_id[
                image_path
            ]
        )


        # No detection
        if len(prediction_df) == 0:

            records.append(
                {
                    "image_id": image_id,
                    "image_path": image_path,
                    "object_name": None,
                    "confidence": None,
                    "xmin": None,
                    "ymin": None,
                    "xmax": None,
                    "ymax": None,
                    "objects_detected": 0
                }
            )

            continue


        # Save detections
        for _, detection in (
            prediction_df.iterrows()
        ):

            records.append(
                {
                    "image_id": image_id,
                    "image_path": image_path,
                    "object_name": detection["name"],
                    "confidence": float(
                        detection["confidence"]
                    ),
                    "xmin": float(
                        detection["xmin"]
                    ),
                    "ymin": float(
                        detection["ymin"]
                    ),
                    "xmax": float(
                        detection["xmax"]
                    ),
                    "ymax": float(
                        detection["ymax"]
                    ),
                    "objects_detected": len(
                        prediction_df
                    )
                }
            )


    del results
    del predictions

    gc.collect()


    if torch.cuda.is_available():

        torch.cuda.empty_cache()


    return records


print(
    "✓ YOLOv5 batch detection function restored"
)

✓ YOLOv5 batch detection function restored


CELL F — Check your existing checkpoints FIRST

In [ ]:
# ============================================================
# CELL F — CHECK EXISTING CHECKPOINTS
# ============================================================

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    )
)


print("=" * 70)
print("EXISTING YOLOv5 CHECKPOINTS")
print("=" * 70)

print(
    "\nCheckpoint files found:",
    len(checkpoint_files)
)


processed_ids = set()


for checkpoint_file in checkpoint_files:

    checkpoint_df = pd.read_csv(
        checkpoint_file
    )


    unique_ids = (
        checkpoint_df[
            "image_id"
        ]
        .dropna()
        .unique()
    )


    processed_ids.update(
        unique_ids
    )


    print(
        f"\n{checkpoint_file.name}"
    )

    print(
        "Unique images:",
        len(unique_ids)
    )


print("\n" + "-" * 70)

print(
    "TOTAL UNIQUE COMPLETED IMAGES:",
    len(processed_ids)
)


print(
    "TOTAL REMAINING:",
    len(all_image_paths)
    - len(processed_ids)
)


assert (
    len(processed_ids)
    == 1536
), (
    "Expected 1,536 completed images "
    "based on the three checkpoints you showed."
)


print(
    "\n✓ CHECKPOINTS VERIFIED — SAFE TO RESUME"
)

EXISTING YOLOv5 CHECKPOINTS

Checkpoint files found: 0

----------------------------------------------------------------------
TOTAL UNIQUE COMPLETED IMAGES: 0
TOTAL REMAINING: 39874


AssertionError: Expected 1,536 completed images based on the three checkpoints you showed.

In [ ]:
# ============================================================
# CELL G — SEARCH PROJECT FOR YOLO CHECKPOINTS
# ============================================================

from pathlib import Path

print("=" * 70)
print("SEARCHING ENTIRE PROJECT FOR YOLO CHECKPOINT FILES")
print("=" * 70)

# Search for CSV files that look like checkpoints
checkpoint_matches = []

for pattern in [
    "checkpoint*.csv",
    "*checkpoint*.csv",
    "*yolo*.csv",
    "*YOLO*.csv"
]:
    checkpoint_matches.extend(
        list(PROJECT_DIR.rglob(pattern))
    )

# Remove duplicates
checkpoint_matches = sorted(
    set(checkpoint_matches)
)

print("\nFiles found:", len(checkpoint_matches))
print("-" * 70)

for file_path in checkpoint_matches:
    print(file_path)

print("\n" + "=" * 70)
print("SEARCH COMPLETE")
print("=" * 70)

SEARCHING ENTIRE PROJECT FOR YOLO CHECKPOINT FILES

Files found: 3
----------------------------------------------------------------------
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints/checkpoint_00001.csv
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints/checkpoint_00002.csv
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints/checkpoint_00003.csv

SEARCH COMPLETE


In [ ]:
# ============================================================
# CELL H — RESTORE CORRECT YOLOv5 CHECKPOINT PATH
# ============================================================

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "Outputs"
    / "YOLOv5_Detection"
    / "Checkpoints"
)

print("=" * 70)
print("CORRECT YOLOv5 CHECKPOINT DIRECTORY")
print("=" * 70)

print("\nCheckpoint directory:")
print(CHECKPOINT_DIR)

print("\nExists:", CHECKPOINT_DIR.exists())


checkpoint_files = sorted(
    CHECKPOINT_DIR.glob("checkpoint_*.csv")
)


processed_ids = set()


print("\nCheckpoint files found:", len(checkpoint_files))
print("-" * 70)


for checkpoint_file in checkpoint_files:

    checkpoint_df = pd.read_csv(checkpoint_file)

    unique_ids = (
        checkpoint_df["image_id"]
        .dropna()
        .unique()
    )

    processed_ids.update(unique_ids)

    print("\nFile:", checkpoint_file.name)
    print("Rows:", len(checkpoint_df))
    print("Unique images:", len(unique_ids))


print("\n" + "-" * 70)

print("TOTAL UNIQUE COMPLETED IMAGES:", len(processed_ids))
print("TOTAL REMAINING:", 39874 - len(processed_ids))


assert len(processed_ids) == 1536, (
    f"Expected 1536 completed images, "
    f"but found {len(processed_ids)}"
)


print("\n✓ ALL 3 CHECKPOINTS VERIFIED")
print("✓ SAFE TO RESUME YOLOv5 PROCESSING")

CORRECT YOLOv5 CHECKPOINT DIRECTORY

Checkpoint directory:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints

Exists: True

Checkpoint files found: 3
----------------------------------------------------------------------

File: checkpoint_00001.csv
Rows: 2082
Unique images: 512

File: checkpoint_00002.csv
Rows: 1910
Unique images: 512

File: checkpoint_00003.csv
Rows: 1961
Unique images: 512

----------------------------------------------------------------------
TOTAL UNIQUE COMPLETED IMAGES: 1536
TOTAL REMAINING: 38338

✓ ALL 3 CHECKPOINTS VERIFIED
✓ SAFE TO RESUME YOLOv5 PROCESSING


In [ ]:
# ============================================================
# CELL I — SPEED TEST ON REMAINING REAL IMAGES
# ============================================================

BATCH_SIZE = 64

# Get only images that are NOT already completed
remaining_image_paths = [
    image_path
    for image_path in all_image_paths
    if path_to_id[str(image_path)] not in processed_ids
]

print("=" * 70)
print("YOLOv5 RESUME SPEED TEST")
print("=" * 70)

print("\nCompleted images:", len(processed_ids))
print("Remaining images:", len(remaining_image_paths))

assert len(remaining_image_paths) == 38338

# Test 64 remaining images
test_paths = remaining_image_paths[:BATCH_SIZE]

torch.cuda.synchronize()
start_time = time.time()

test_records = run_yolov5_batch(test_paths)

torch.cuda.synchronize()
elapsed = time.time() - start_time

print("\nImages tested:", len(test_paths))
print("Detection records:", len(test_records))
print(f"Time taken: {elapsed:.2f} seconds")
print(f"Speed: {len(test_paths) / elapsed:.2f} images/second")

print("\n✓ RESUME SPEED TEST COMPLETE")

YOLOv5 RESUME SPEED TEST

Completed images: 1536
Remaining images: 38338

Images tested: 64
Detection records: 232
Time taken: 28.29 seconds
Speed: 2.26 images/second

✓ RESUME SPEED TEST COMPLETE


In [ ]:
# ============================================================
# CELL J — TEST YOLOv5 SPEED FROM LOCAL COLAB STORAGE
# ============================================================

import shutil
from pathlib import Path
import time
import torch


print("=" * 70)
print("LOCAL STORAGE YOLOv5 SPEED TEST")
print("=" * 70)


# ------------------------------------------------------------
# LOCAL TEMPORARY DIRECTORY
# ------------------------------------------------------------

LOCAL_TEST_DIR = Path(
    "/content/yolo_local_speed_test"
)


# Remove old test files if they exist
if LOCAL_TEST_DIR.exists():

    shutil.rmtree(
        LOCAL_TEST_DIR
    )


LOCAL_TEST_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# SELECT 64 REMAINING IMAGES
# ------------------------------------------------------------

test_drive_paths = (
    remaining_image_paths[:64]
)


print(
    "\nImages selected:",
    len(test_drive_paths)
)


# ------------------------------------------------------------
# COPY FROM GOOGLE DRIVE TO LOCAL COLAB STORAGE
# ------------------------------------------------------------

print(
    "\nCopying images from Drive to local storage..."
)


local_test_paths = []


copy_start = time.time()


for index, drive_path in enumerate(
    test_drive_paths
):

    drive_path = Path(
        drive_path
    )


    # Give every copied file a unique name
    local_path = (
        LOCAL_TEST_DIR
        / f"{index}_{drive_path.name}"
    )


    shutil.copy2(
        drive_path,
        local_path
    )


    local_test_paths.append(
        str(local_path)
    )


copy_time = (
    time.time()
    - copy_start
)


print(
    f"Copy time: {copy_time:.2f} seconds"
)


# ------------------------------------------------------------
# GPU INFERENCE TEST
# ------------------------------------------------------------

print(
    "\nRunning YOLOv5 from LOCAL storage..."
)


# Small warm-up
with torch.no_grad():

    _ = yolo_model(
        local_test_paths[:4],
        size=IMAGE_SIZE
    )


if torch.cuda.is_available():

    torch.cuda.synchronize()


start_time = time.time()


with torch.no_grad():

    results = yolo_model(
        local_test_paths,
        size=IMAGE_SIZE
    )


# Force results to complete
predictions = (
    results
    .pandas()
    .xyxy
)


if torch.cuda.is_available():

    torch.cuda.synchronize()


inference_time = (
    time.time()
    - start_time
)


# ------------------------------------------------------------
# RESULTS
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("LOCAL STORAGE SPEED RESULTS")
print("=" * 70)

print(
    "\nImages:",
    len(local_test_paths)
)

print(
    f"Copy time from Drive: {copy_time:.2f} seconds"
)

print(
    f"YOLO inference time: {inference_time:.2f} seconds"
)

print(
    f"Inference speed: "
    f"{len(local_test_paths) / inference_time:.2f} "
    f"images/second"
)


print(
    "\nOriginal Drive speed: 2.26 images/second"
)


print(
    "\n✓ LOCAL SPEED TEST COMPLETE"
)

LOCAL STORAGE YOLOv5 SPEED TEST

Images selected: 64

Copying images from Drive to local storage...
Copy time: 0.37 seconds

Running YOLOv5 from LOCAL storage...

LOCAL STORAGE SPEED RESULTS

Images: 64
Copy time from Drive: 0.37 seconds
YOLO inference time: 1.14 seconds
Inference speed: 56.05 images/second

Original Drive speed: 2.26 images/second

✓ LOCAL SPEED TEST COMPLETE


In [ ]:
# ============================================================
# CELL K — COPY IMAGE DATASETS TO LOCAL COLAB STORAGE
# ============================================================

import shutil
import time
from pathlib import Path


print("=" * 70)
print("COPYING IMAGE DATASETS TO LOCAL COLAB STORAGE")
print("=" * 70)


# ------------------------------------------------------------
# LOCAL ROOT
# ------------------------------------------------------------

LOCAL_DATASET_DIR = Path(
    "/content/Image_captioning_Local"
)

LOCAL_FLICKR8K_DIR = (
    LOCAL_DATASET_DIR
    / "Flickr8K"
)

LOCAL_FLICKR30K_DIR = (
    LOCAL_DATASET_DIR
    / "Flickr30K"
)


LOCAL_DATASET_DIR.mkdir(
    parents=True,
    exist_ok=True
)


# ------------------------------------------------------------
# SOURCE DIRECTORIES
# ------------------------------------------------------------

DRIVE_FLICKR8K_IMAGES = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Dataset/Raw/Flickr8K/Images"
)


# Flickr30K images are inside the nested folder
DRIVE_FLICKR30K_ROOT = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Dataset/Raw/Flickr30k"
)


print("\nFlickr8K source:")
print(DRIVE_FLICKR8K_IMAGES)

print("Exists:", DRIVE_FLICKR8K_IMAGES.exists())


print("\nFlickr30K source:")
print(DRIVE_FLICKR30K_ROOT)

print("Exists:", DRIVE_FLICKR30K_ROOT.exists())


assert DRIVE_FLICKR8K_IMAGES.exists(), (
    "Flickr8K image directory not found."
)

assert DRIVE_FLICKR30K_ROOT.exists(), (
    "Flickr30K root directory not found."
)


# ------------------------------------------------------------
# FIND FLICKR30K JPG DIRECTORY AUTOMATICALLY
# ------------------------------------------------------------

flickr30k_jpg_files = list(
    DRIVE_FLICKR30K_ROOT.rglob("*.jpg")
)

assert len(flickr30k_jpg_files) > 0, (
    "No Flickr30K JPG images found."
)


# Determine the common image directory structure
DRIVE_FLICKR30K_IMAGES = (
    flickr30k_jpg_files[0]
    .parent
)


print("\nFlickr30K image directory detected:")
print(DRIVE_FLICKR30K_IMAGES)


# ------------------------------------------------------------
# COPY FUNCTION
# ------------------------------------------------------------

def copy_dataset_if_needed(
    source_dir,
    destination_dir,
    expected_count,
    dataset_name
):

    destination_dir.mkdir(
        parents=True,
        exist_ok=True
    )


    existing_files = list(
        destination_dir.glob("*.jpg")
    )


    print("\n" + "-" * 70)
    print(dataset_name)
    print("-" * 70)

    print(
        "Images already local:",
        len(existing_files)
    )


    if len(existing_files) >= expected_count:

        print(
            "✓ Dataset already copied."
        )

        return


    print(
        "Copying missing images..."
    )


    source_files = list(
        source_dir.glob("*.jpg")
    )


    copied = 0

    start_time = time.time()


    for source_file in tqdm(
        source_files,
        desc=f"Copying {dataset_name}"
    ):

        destination_file = (
            destination_dir
            / source_file.name
        )


        if destination_file.exists():

            continue


        shutil.copy2(
            source_file,
            destination_file
        )


        copied += 1


    elapsed = (
        time.time()
        - start_time
    )


    final_count = len(
        list(
            destination_dir.glob("*.jpg")
        )
    )


    print(
        f"\nNew images copied: {copied}"
    )

    print(
        f"Total local images: {final_count}"
    )

    print(
        f"Time: {elapsed / 60:.2f} minutes"
    )


# ------------------------------------------------------------
# COPY FLICKR8K
# ------------------------------------------------------------

copy_dataset_if_needed(
    source_dir=DRIVE_FLICKR8K_IMAGES,
    destination_dir=LOCAL_FLICKR8K_DIR,
    expected_count=8091,
    dataset_name="Flickr8K"
)


# ------------------------------------------------------------
# COPY FLICKR30K
# ------------------------------------------------------------

copy_dataset_if_needed(
    source_dir=DRIVE_FLICKR30K_IMAGES,
    destination_dir=LOCAL_FLICKR30K_DIR,
    expected_count=31783,
    dataset_name="Flickr30K"
)


# ------------------------------------------------------------
# FINAL VERIFICATION
# ------------------------------------------------------------

local_flickr8k_count = len(
    list(
        LOCAL_FLICKR8K_DIR.glob("*.jpg")
    )
)

local_flickr30k_count = len(
    list(
        LOCAL_FLICKR30K_DIR.glob("*.jpg")
    )
)


print("\n" + "=" * 70)
print("LOCAL DATASET VERIFICATION")
print("=" * 70)

print(
    "\nFlickr8K local images:",
    local_flickr8k_count
)

print(
    "Expected:",
    8091
)


print(
    "\nFlickr30K local images:",
    local_flickr30k_count
)

print(
    "Expected:",
    31783
)


assert local_flickr8k_count == 8091
assert local_flickr30k_count == 31783


print(
    "\n✓ ALL 39,874 IMAGES AVAILABLE LOCALLY"
)

print(
    "✓ READY FOR HIGH-SPEED YOLO PROCESSING"
)

COPYING IMAGE DATASETS TO LOCAL COLAB STORAGE

Flickr8K source:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr8K/Images
Exists: True

Flickr30K source:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k
Exists: True

Flickr30K image directory detected:
/content/drive/MyDrive/Image_captioning/Dataset/Raw/Flickr30k/flickr30k/flickr30k_images

----------------------------------------------------------------------
Flickr8K
----------------------------------------------------------------------
Images already local: 0
Copying missing images...


Copying Flickr8K:   0%|          | 0/8091 [00:00<?, ?it/s]


New images copied: 8091
Total local images: 8091
Time: 4.52 minutes

----------------------------------------------------------------------
Flickr30K
----------------------------------------------------------------------
Images already local: 0
Copying missing images...


Copying Flickr30K:   0%|          | 0/31783 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [ ]:
# ============================================================
# CELL K2 — FAST FLICKR30K EXTRACTION TO LOCAL COLAB STORAGE
# ============================================================

import shutil
import subprocess
from pathlib import Path
import time
import os


print("=" * 70)
print("FAST FLICKR30K LOCAL EXTRACTION")
print("=" * 70)


# ------------------------------------------------------------
# PATHS
# ------------------------------------------------------------

FLICKR30K_ZIP = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Dataset/ZIP/flickr30k.zip"
)

LOCAL_FLICKR30K_ROOT = Path(
    "/content/Image_captioning_Local/Flickr30K"
)


print("\nZIP file:")
print(FLICKR30K_ZIP)

print("\nZIP exists:")
print(FLICKR30K_ZIP.exists())


assert FLICKR30K_ZIP.exists(), (
    "Flickr30K ZIP file not found."
)


# ------------------------------------------------------------
# CHECK AVAILABLE LOCAL DISK SPACE
# ------------------------------------------------------------

print("\n" + "-" * 70)
print("LOCAL DISK SPACE")
print("-" * 70)

subprocess.run(
    ["df", "-h", "/content"]
)


# ------------------------------------------------------------
# REMOVE PARTIAL COPY
# ------------------------------------------------------------

if LOCAL_FLICKR30K_ROOT.exists():

    print(
        "\nRemoving partial Flickr30K copy..."
    )

    shutil.rmtree(
        LOCAL_FLICKR30K_ROOT
    )


LOCAL_FLICKR30K_ROOT.mkdir(
    parents=True,
    exist_ok=True
)


print(
    "\n✓ Clean local Flickr30K directory ready"
)


# ------------------------------------------------------------
# EXTRACT ZIP DIRECTLY TO LOCAL STORAGE
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("EXTRACTING FLICKR30K ZIP TO LOCAL STORAGE")
print("=" * 70)

print(
    "\nThis is much faster than copying "
    "31,783 files individually."
)


start_time = time.time()


result = subprocess.run(
    [
        "unzip",
        "-q",
        str(FLICKR30K_ZIP),
        "-d",
        str(LOCAL_FLICKR30K_ROOT)
    ],
    stdout=subprocess.PIPE,
    stderr=subprocess.PIPE,
    text=True
)


elapsed = (
    time.time()
    - start_time
)


print(
    f"\nExtraction time: {elapsed / 60:.2f} minutes"
)


if result.returncode != 0:

    print("\nUNZIP ERROR:")
    print(result.stderr)

else:

    print(
        "\n✓ ZIP EXTRACTION COMPLETED"
    )


# ------------------------------------------------------------
# LOCATE ALL EXTRACTED JPG FILES
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("VERIFYING EXTRACTED FLICKR30K IMAGES")
print("=" * 70)


jpg_files = list(
    LOCAL_FLICKR30K_ROOT.rglob("*.jpg")
)


print(
    "\nTotal JPG images found:",
    len(jpg_files)
)


if len(jpg_files) > 0:

    print(
        "\nSample image:"
    )

    print(
        jpg_files[0]
    )


assert len(jpg_files) >= 31783, (
    f"Expected at least 31,783 images, "
    f"but found {len(jpg_files)}"
)


print("\n" + "=" * 70)
print("✓ FLICKR30K READY LOCALLY")
print("=" * 70)

print(
    "\nImages:",
    len(jpg_files)
)

print(
    "\nLocal root:"
)

print(
    LOCAL_FLICKR30K_ROOT
)

FAST FLICKR30K LOCAL EXTRACTION

ZIP file:
/content/drive/MyDrive/Image_captioning/Dataset/ZIP/flickr30k.zip

ZIP exists:
True

----------------------------------------------------------------------
LOCAL DISK SPACE
----------------------------------------------------------------------

Removing partial Flickr30K copy...

✓ Clean local Flickr30K directory ready

EXTRACTING FLICKR30K ZIP TO LOCAL STORAGE

This is much faster than copying 31,783 files individually.

Extraction time: 3.83 minutes

✓ ZIP EXTRACTION COMPLETED

VERIFYING EXTRACTED FLICKR30K IMAGES

Total JPG images found: 63566

Sample image:
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1177949654.jpg

✓ FLICKR30K READY LOCALLY

Images: 63566

Local root:
/content/Image_captioning_Local/Flickr30K


In [ ]:
# ============================================================
# CELL L — VERIFY LOCAL DATA + RESTORE YOLO RESUME STATE
# ============================================================

import pandas as pd
from pathlib import Path
import glob


print("=" * 70)
print("RESTORING YOLOv5 RESUME STATE")
print("=" * 70)


# ============================================================
# 1. IMPORTANT PROJECT PATHS
# ============================================================

CAPTIONS_FILE = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Dataset/Processed/Captions/"
    "captions_processed.csv"
)

CHECKPOINT_DIR = Path(
    "/content/drive/MyDrive/Image_captioning/"
    "Outputs/YOLOv5_Detection/"
    "Checkpoints"
)

LOCAL_FLICKR8K_DIR = Path(
    "/content/Image_captioning_Local/Flickr8K"
)

LOCAL_FLICKR30K_ROOT = Path(
    "/content/Image_captioning_Local/Flickr30K"
)


# ============================================================
# 2. VERIFY IMPORTANT FILES
# ============================================================

print("\nCAPTIONS FILE:")
print(CAPTIONS_FILE)
print("Exists:", CAPTIONS_FILE.exists())

print("\nCHECKPOINT DIRECTORY:")
print(CHECKPOINT_DIR)
print("Exists:", CHECKPOINT_DIR.exists())

print("\nLOCAL FLICKR8K:")
print(LOCAL_FLICKR8K_DIR)
print("Exists:", LOCAL_FLICKR8K_DIR.exists())

print("\nLOCAL FLICKR30K:")
print(LOCAL_FLICKR30K_ROOT)
print("Exists:", LOCAL_FLICKR30K_ROOT.exists())


assert CAPTIONS_FILE.exists(), "Processed captions file not found."
assert CHECKPOINT_DIR.exists(), "YOLO checkpoint directory not found."
assert LOCAL_FLICKR8K_DIR.exists(), "Local Flickr8K not found."
assert LOCAL_FLICKR30K_ROOT.exists(), "Local Flickr30K not found."


# ============================================================
# 3. LOAD UNIQUE IMAGE MANIFEST
# ============================================================

print("\n" + "=" * 70)
print("LOADING IMAGE MANIFEST")
print("=" * 70)


captions_df = pd.read_csv(
    CAPTIONS_FILE
)


required_columns = {
    "image_id",
    "dataset",
    "image_name"
}


assert required_columns.issubset(
    captions_df.columns
), (
    "Required columns missing from captions file."
)


image_manifest = (
    captions_df[
        [
            "image_id",
            "dataset",
            "image_name"
        ]
    ]
    .drop_duplicates()
    .reset_index(drop=True)
)


print("\nTotal unique images:")
print(len(image_manifest))

print("\nDataset distribution:")
print(
    image_manifest["dataset"]
    .value_counts()
)


assert len(image_manifest) == 39874, (
    f"Expected 39874 images, found {len(image_manifest)}"
)


# ============================================================
# 4. LOCATE LOCAL FLICKR8K IMAGES
# ============================================================

print("\n" + "=" * 70)
print("VERIFYING LOCAL FLICKR8K")
print("=" * 70)


local_flickr8k_files = list(
    LOCAL_FLICKR8K_DIR.glob("*.jpg")
)


print("\nLocal Flickr8K images:")
print(len(local_flickr8k_files))


assert len(local_flickr8k_files) == 8091, (
    f"Expected 8091 Flickr8K images, "
    f"found {len(local_flickr8k_files)}"
)


# ============================================================
# 5. LOCATE LOCAL FLICKR30K IMAGES
# ============================================================

print("\n" + "=" * 70)
print("LOCATING LOCAL FLICKR30K IMAGES")
print("=" * 70)


local_flickr30k_files = list(
    LOCAL_FLICKR30K_ROOT.rglob("*.jpg")
)


print("\nLocal Flickr30K images:")
print(len(local_flickr30k_files))


assert len(local_flickr30k_files) >= 31783, (
    f"Expected at least 31783 Flickr30K images, "
    f"found {len(local_flickr30k_files)}"
)


print("\nSample Flickr30K path:")

print(
    local_flickr30k_files[0]
)


# ============================================================
# 6. BUILD FAST FILENAME → LOCAL PATH MAPS
# ============================================================

print("\n" + "=" * 70)
print("BUILDING LOCAL IMAGE PATH MAPS")
print("=" * 70)


flickr8k_path_map = {
    path.name: str(path)
    for path in local_flickr8k_files
}


flickr30k_path_map = {
    path.name: str(path)
    for path in local_flickr30k_files
}


print("\nFlickr8K mapped files:")
print(len(flickr8k_path_map))

print("\nFlickr30K mapped files:")
print(len(flickr30k_path_map))


# ============================================================
# 7. ASSIGN LOCAL PATH TO EVERY IMAGE
# ============================================================

def get_local_image_path(row):

    if row["dataset"] == "Flickr8K":

        return flickr8k_path_map.get(
            row["image_name"]
        )

    elif row["dataset"] == "Flickr30K":

        return flickr30k_path_map.get(
            row["image_name"]
        )

    return None


image_manifest[
    "local_image_path"
] = image_manifest.apply(
    get_local_image_path,
    axis=1
)


missing_local_images = (
    image_manifest[
        "local_image_path"
    ]
    .isna()
    .sum()
)


print("\nImages missing locally:")
print(missing_local_images)


assert missing_local_images == 0, (
    "Some manifest images could not be found locally."
)


print(
    "\n✓ ALL MANIFEST IMAGES HAVE LOCAL PATHS"
)


# ============================================================
# 8. LOAD EXISTING YOLO CHECKPOINTS
# ============================================================

print("\n" + "=" * 70)
print("LOADING EXISTING YOLO CHECKPOINTS")
print("=" * 70)


checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    )
)


print("\nCheckpoint files found:")
print(len(checkpoint_files))


for checkpoint in checkpoint_files:

    print(
        checkpoint.name
    )


assert len(checkpoint_files) == 3, (
    f"Expected 3 checkpoints, found "
    f"{len(checkpoint_files)}"
)


processed_ids = set()


for checkpoint_file in checkpoint_files:

    checkpoint_df = pd.read_csv(
        checkpoint_file
    )


    assert "image_id" in checkpoint_df.columns, (
        f"image_id column missing in "
        f"{checkpoint_file.name}"
    )


    processed_ids.update(
        checkpoint_df[
            "image_id"
        ]
        .dropna()
        .unique()
    )


print("\nTotal completed unique images:")
print(len(processed_ids))


assert len(processed_ids) == 1536, (
    f"Expected 1536 completed images, "
    f"found {len(processed_ids)}"
)


print(
    "\n✓ PREVIOUS YOLO WORK RESTORED"
)


# ============================================================
# 9. CREATE REMAINING LOCAL IMAGE LIST
# ============================================================

remaining_manifest = (
    image_manifest[
        ~image_manifest["image_id"]
        .isin(processed_ids)
    ]
    .reset_index(drop=True)
)


remaining_image_paths = (
    remaining_manifest[
        "local_image_path"
    ]
    .tolist()
)


remaining_image_ids = (
    remaining_manifest[
        "image_id"
    ]
    .tolist()
)


print("\n" + "=" * 70)
print("FINAL RESUME STATUS")
print("=" * 70)


print("\nTotal dataset images:")
print(len(image_manifest))

print("\nAlready completed:")
print(len(processed_ids))

print("\nRemaining:")
print(len(remaining_image_paths))


assert len(remaining_image_paths) == 38338, (
    f"Expected 38338 remaining images, "
    f"found {len(remaining_image_paths)}"
)


assert (
    len(remaining_image_paths)
    ==
    len(remaining_image_ids)
)


print("\nNext checkpoint number:")
print("checkpoint_00004.csv")


print("\n" + "=" * 70)
print("✓ LOCAL DATA VERIFIED")
print("✓ 1,536 PREVIOUS IMAGES WILL NOT BE REPEATED")
print("✓ 38,338 LOCAL IMAGES READY FOR FAST YOLO")
print("=" * 70)

RESTORING YOLOv5 RESUME STATE

CAPTIONS FILE:
/content/drive/MyDrive/Image_captioning/Dataset/Processed/Captions/captions_processed.csv
Exists: True

CHECKPOINT DIRECTORY:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints
Exists: True

LOCAL FLICKR8K:
/content/Image_captioning_Local/Flickr8K
Exists: True

LOCAL FLICKR30K:
/content/Image_captioning_Local/Flickr30K
Exists: True

LOADING IMAGE MANIFEST

Total unique images:
39874

Dataset distribution:
dataset
Flickr30K    31783
Flickr8K      8091
Name: count, dtype: int64

VERIFYING LOCAL FLICKR8K

Local Flickr8K images:
8091

LOCATING LOCAL FLICKR30K IMAGES

Local Flickr30K images:
63566

Sample Flickr30K path:
/content/Image_captioning_Local/Flickr30K/flickr30k_images/flickr30k_images/1177949654.jpg

BUILDING LOCAL IMAGE PATH MAPS

Flickr8K mapped files:
8091

Flickr30K mapped files:
31783

Images missing locally:
0

✓ ALL MANIFEST IMAGES HAVE LOCAL PATHS

LOADING EXISTING YOLO CHECKPOINTS

Checkpoint files f

In [ ]:
# ============================================================
# CELL M — FAST YOLOv5 FULL DATASET RESUME
# ============================================================

import pandas as pd
import torch
import time
import gc
from pathlib import Path
from tqdm.auto import tqdm


print("=" * 70)
print("YOLOv5 FAST FULL DATASET RESUME")
print("=" * 70)


# ============================================================
# 1. VERIFY MODEL
# ============================================================

assert "yolo_model" in globals(), (
    "YOLOv5 model is not loaded. "
    "Run the YOLOv5 model loading cell first."
)


print("\nYOLOv5 model: READY")

print(
    "GPU:",
    torch.cuda.get_device_name()
    if torch.cuda.is_available()
    else "CPU"
)


# ============================================================
# 2. PROCESSING CONFIGURATION
# ============================================================

BATCH_SIZE = 64

CHECKPOINT_INTERVAL = 512

IMAGE_SIZE = 640


print("\nProcessing configuration:")

print("Batch size:", BATCH_SIZE)

print(
    "Checkpoint interval:",
    CHECKPOINT_INTERVAL,
    "images"
)

print("Image size:", IMAGE_SIZE)


# ============================================================
# 3. VERIFY RESUME DATA
# ============================================================

assert len(remaining_manifest) == 38338, (
    f"Expected 38338 remaining images, "
    f"found {len(remaining_manifest)}"
)


assert len(processed_ids) == 1536, (
    f"Expected 1536 processed images, "
    f"found {len(processed_ids)}"
)


assert CHECKPOINT_DIR.exists(), (
    "Checkpoint directory not found."
)


print("\nAlready completed:", len(processed_ids))

print("Remaining images:", len(remaining_manifest))


# ============================================================
# 4. FIND NEXT CHECKPOINT NUMBER
# ============================================================

existing_checkpoints = sorted(
    CHECKPOINT_DIR.glob("checkpoint_*.csv")
)


checkpoint_numbers = []


for checkpoint_file in existing_checkpoints:

    try:

        number = int(
            checkpoint_file.stem
            .split("_")[-1]
        )

        checkpoint_numbers.append(
            number
        )

    except Exception:

        pass


next_checkpoint_number = (
    max(checkpoint_numbers) + 1
    if checkpoint_numbers
    else 1
)


print(
    "\nStarting checkpoint:",
    f"checkpoint_{next_checkpoint_number:05d}.csv"
)


# ============================================================
# 5. PROCESS ONE BATCH
# ============================================================

def process_yolo_batch(batch_df):

    batch_paths = (
        batch_df[
            "local_image_path"
        ]
        .tolist()
    )


    # --------------------------------------------------------
    # YOLO INFERENCE
    # --------------------------------------------------------

    with torch.no_grad():

        results = yolo_model(
            batch_paths,
            size=IMAGE_SIZE
        )


    # One DataFrame per image
    predictions = (
        results
        .pandas()
        .xyxy
    )


    batch_records = []


    # --------------------------------------------------------
    # PROCESS EACH IMAGE
    # --------------------------------------------------------

    for row_index, (
        _,
        image_row
    ) in enumerate(
        batch_df.iterrows()
    ):

        image_id = (
            image_row["image_id"]
        )

        image_path = (
            image_row["local_image_path"]
        )

        detections = (
            predictions[row_index]
        )


        detection_count = (
            len(detections)
        )


        # ----------------------------------------------------
        # CASE 1 — NO DETECTIONS
        # ----------------------------------------------------

        if detection_count == 0:

            batch_records.append(
                {
                    "image_id": image_id,
                    "image_path": image_path,
                    "object_name": None,
                    "confidence": None,
                    "xmin": None,
                    "ymin": None,
                    "xmax": None,
                    "ymax": None,
                    "objects_detected": 0
                }
            )


        # ----------------------------------------------------
        # CASE 2 — DETECTIONS FOUND
        # ----------------------------------------------------

        else:

            for _, detection in (
                detections.iterrows()
            ):

                batch_records.append(
                    {
                        "image_id": image_id,
                        "image_path": image_path,
                        "object_name": detection["name"],
                        "confidence": float(
                            detection["confidence"]
                        ),
                        "xmin": float(
                            detection["xmin"]
                        ),
                        "ymin": float(
                            detection["ymin"]
                        ),
                        "xmax": float(
                            detection["xmax"]
                        ),
                        "ymax": float(
                            detection["ymax"]
                        ),
                        "objects_detected": detection_count
                    }
                )


    return batch_records


# ============================================================
# 6. FULL PROCESSING
# ============================================================

print("\n" + "=" * 70)
print("STARTING YOLOv5 PROCESSING")
print("=" * 70)


total_remaining = len(
    remaining_manifest
)


completed_this_run = 0

checkpoint_records = []

checkpoint_image_count = 0

checkpoint_number = (
    next_checkpoint_number
)


start_time = time.time()


# ------------------------------------------------------------
# MAIN PROGRESS BAR
# ------------------------------------------------------------

progress_bar = tqdm(
    total=total_remaining,
    desc="YOLOv5 Detection",
    unit="images"
)


# ------------------------------------------------------------
# BATCH LOOP
# ------------------------------------------------------------

for batch_start in range(
    0,
    total_remaining,
    BATCH_SIZE
):

    batch_end = min(
        batch_start + BATCH_SIZE,
        total_remaining
    )


    batch_df = (
        remaining_manifest
        .iloc[
            batch_start:batch_end
        ]
    )


    # --------------------------------------------------------
    # RUN YOLO
    # --------------------------------------------------------

    batch_records = (
        process_yolo_batch(
            batch_df
        )
    )


    # --------------------------------------------------------
    # ADD RECORDS TO CURRENT CHECKPOINT
    # --------------------------------------------------------

    checkpoint_records.extend(
        batch_records
    )


    batch_size_actual = (
        len(batch_df)
    )


    checkpoint_image_count += (
        batch_size_actual
    )


    completed_this_run += (
        batch_size_actual
    )


    progress_bar.update(
        batch_size_actual
    )


    # --------------------------------------------------------
    # CALCULATE SPEED + ETA
    # --------------------------------------------------------

    elapsed = (
        time.time()
        - start_time
    )


    speed = (
        completed_this_run / elapsed
        if elapsed > 0
        else 0
    )


    remaining_count = (
        total_remaining
        - completed_this_run
    )


    eta_seconds = (
        remaining_count / speed
        if speed > 0
        else 0
    )


    progress_bar.set_postfix(
        {
            "speed": (
                f"{speed:.1f} img/s"
            ),
            "ETA": (
                f"{eta_seconds / 60:.1f} min"
            )
        }
    )


    # ========================================================
    # SAVE CHECKPOINT
    # ========================================================

    if (
        checkpoint_image_count
        >= CHECKPOINT_INTERVAL
    ):

        checkpoint_df = (
            pd.DataFrame(
                checkpoint_records
            )
        )


        checkpoint_path = (
            CHECKPOINT_DIR
            /
            f"checkpoint_{checkpoint_number:05d}.csv"
        )


        checkpoint_df.to_csv(
            checkpoint_path,
            index=False
        )


        unique_checkpoint_images = (
            checkpoint_df[
                "image_id"
            ]
            .nunique()
        )


        print(
            "\n✓ Checkpoint saved:",
            checkpoint_path.name
        )

        print(
            "  Unique images:",
            unique_checkpoint_images
        )

        print(
            "  Detection records:",
            len(checkpoint_df)
        )


        # ----------------------------------------------------
        # RESET CHECKPOINT MEMORY
        # ----------------------------------------------------

        checkpoint_records = []

        checkpoint_image_count = 0

        checkpoint_number += 1


        del checkpoint_df

        gc.collect()


# ============================================================
# 7. SAVE FINAL PARTIAL CHECKPOINT
# ============================================================

if len(checkpoint_records) > 0:

    checkpoint_df = (
        pd.DataFrame(
            checkpoint_records
        )
    )


    checkpoint_path = (
        CHECKPOINT_DIR
        /
        f"checkpoint_{checkpoint_number:05d}.csv"
    )


    checkpoint_df.to_csv(
        checkpoint_path,
        index=False
    )


    unique_checkpoint_images = (
        checkpoint_df[
            "image_id"
        ]
        .nunique()
    )


    print(
        "\n✓ Final checkpoint saved:",
        checkpoint_path.name
    )

    print(
        "  Unique images:",
        unique_checkpoint_images
    )

    print(
        "  Detection records:",
        len(checkpoint_df)
    )


# ============================================================
# 8. FINAL SUMMARY
# ============================================================

progress_bar.close()


total_elapsed = (
    time.time()
    - start_time
)


final_speed = (
    completed_this_run
    / total_elapsed
)


print("\n" + "=" * 70)
print("YOLOv5 PROCESSING COMPLETE")
print("=" * 70)


print(
    "\nPreviously completed:",
    len(processed_ids)
)

print(
    "Processed in this run:",
    completed_this_run
)

print(
    "Total dataset images:",
    len(processed_ids)
    +
    completed_this_run
)


print(
    f"\nTotal processing time: "
    f"{total_elapsed / 60:.2f} minutes"
)


print(
    f"Average speed: "
    f"{final_speed:.2f} images/second"
)


print(
    "\nCheckpoints saved in:"
)

print(
    CHECKPOINT_DIR
)


print("\n" + "=" * 70)
print("✓ YOLOv5 FULL DATASET DETECTION FINISHED")
print("=" * 70)

YOLOv5 FAST FULL DATASET RESUME

YOLOv5 model: READY
GPU: Tesla T4

Processing configuration:
Batch size: 64
Checkpoint interval: 512 images
Image size: 640

Already completed: 1536
Remaining images: 38338

Starting checkpoint: checkpoint_00004.csv

STARTING YOLOv5 PROCESSING


YOLOv5 Detection:   0%|          | 0/38338 [00:00<?, ?images/s]


✓ Checkpoint saved: checkpoint_00004.csv
  Unique images: 512
  Detection records: 2348

✓ Checkpoint saved: checkpoint_00005.csv
  Unique images: 512
  Detection records: 1926

✓ Checkpoint saved: checkpoint_00006.csv
  Unique images: 512
  Detection records: 2090

✓ Checkpoint saved: checkpoint_00007.csv
  Unique images: 512
  Detection records: 2163

✓ Checkpoint saved: checkpoint_00008.csv
  Unique images: 512
  Detection records: 2530

✓ Checkpoint saved: checkpoint_00009.csv
  Unique images: 512
  Detection records: 2498

✓ Checkpoint saved: checkpoint_00010.csv
  Unique images: 512
  Detection records: 2536

✓ Checkpoint saved: checkpoint_00011.csv
  Unique images: 512
  Detection records: 2464

✓ Checkpoint saved: checkpoint_00012.csv
  Unique images: 512
  Detection records: 2576

✓ Checkpoint saved: checkpoint_00013.csv
  Unique images: 512
  Detection records: 2283

✓ Checkpoint saved: checkpoint_00014.csv
  Unique images: 512
  Detection records: 2511

✓ Checkpoint saved: 

In [ ]:
# ============================================================
# CELL N — MERGE AND VERIFY ALL YOLOv5 CHECKPOINTS
# ============================================================

import pandas as pd
from pathlib import Path


print("=" * 70)
print("MERGING YOLOv5 CHECKPOINTS")
print("=" * 70)


# ============================================================
# PATHS
# ============================================================

PROJECT_DIR = Path(
    "/content/drive/MyDrive/Image_captioning"
)

CHECKPOINT_DIR = (
    PROJECT_DIR
    / "Outputs"
    / "YOLOv5_Detection"
    / "Checkpoints"
)

FINAL_OUTPUT_DIR = (
    PROJECT_DIR
    / "Outputs"
    / "YOLOv5_Detection"
    / "Final"
)

FINAL_OUTPUT_DIR.mkdir(
    parents=True,
    exist_ok=True
)


FINAL_YOLO_FILE = (
    FINAL_OUTPUT_DIR
    / "yolov5_detections_full.csv"
)


print("\nCheckpoint directory:")
print(CHECKPOINT_DIR)

print("\nFinal output:")
print(FINAL_YOLO_FILE)


# ============================================================
# FIND CHECKPOINTS
# ============================================================

checkpoint_files = sorted(
    CHECKPOINT_DIR.glob(
        "checkpoint_*.csv"
    )
)


print("\n" + "=" * 70)
print("CHECKPOINT FILES FOUND")
print("=" * 70)

print("\nTotal checkpoint files:")
print(len(checkpoint_files))


assert len(checkpoint_files) > 0, (
    "No checkpoint files found."
)


# ============================================================
# LOAD AND MERGE
# ============================================================

all_checkpoint_dfs = []


for checkpoint_file in checkpoint_files:

    checkpoint_df = pd.read_csv(
        checkpoint_file
    )

    print(
        f"\n{checkpoint_file.name}"
    )

    print(
        "  Rows:",
        len(checkpoint_df)
    )

    print(
        "  Unique images:",
        checkpoint_df[
            "image_id"
        ].nunique()
    )

    all_checkpoint_dfs.append(
        checkpoint_df
    )


print("\n" + "=" * 70)
print("MERGING DATA")
print("=" * 70)


yolo_df = pd.concat(
    all_checkpoint_dfs,
    ignore_index=True
)


print(
    "\nTotal detection records before cleanup:",
    len(yolo_df)
)

print(
    "Total unique images:",
    yolo_df[
        "image_id"
    ].nunique()
)


# ============================================================
# REMOVE EXACT DUPLICATE RECORDS
# ============================================================

rows_before = len(yolo_df)


yolo_df = (
    yolo_df
    .drop_duplicates()
    .reset_index(drop=True)
)


rows_after = len(yolo_df)


print(
    "\nDuplicate records removed:",
    rows_before - rows_after
)


# ============================================================
# FINAL DATASET VERIFICATION
# ============================================================

print("\n" + "=" * 70)
print("FINAL YOLOv5 VERIFICATION")
print("=" * 70)


total_unique_images = (
    yolo_df[
        "image_id"
    ]
    .nunique()
)


images_with_no_detection = (
    yolo_df[
        yolo_df[
            "objects_detected"
        ] == 0
    ][
        "image_id"
    ]
    .nunique()
)


images_with_detection = (
    total_unique_images
    - images_with_no_detection
)


print(
    "\nTotal unique images:",
    total_unique_images
)

print(
    "Expected images:",
    39874
)

print(
    "\nImages with detections:",
    images_with_detection
)

print(
    "Images with zero detections:",
    images_with_no_detection
)

print(
    "\nTotal detection records:",
    len(yolo_df)
)


assert total_unique_images == 39874, (
    f"Expected 39874 unique images, "
    f"found {total_unique_images}"
)


# ============================================================
# SORT FOR CLEAN OUTPUT
# ============================================================

yolo_df = (
    yolo_df
    .sort_values(
        by=[
            "image_id",
            "confidence"
        ],
        ascending=[
            True,
            False
        ],
        na_position="last"
    )
    .reset_index(drop=True)
)


# ============================================================
# SAVE FINAL FILE
# ============================================================

yolo_df.to_csv(
    FINAL_YOLO_FILE,
    index=False
)


print("\n" + "=" * 70)
print("FINAL FILE SAVED")
print("=" * 70)

print(
    "\nPath:"
)

print(
    FINAL_YOLO_FILE
)

print(
    "\nFinal shape:",
    yolo_df.shape
)


print("\nColumns:")

print(
    list(yolo_df.columns)
)


print("\n" + "=" * 70)
print("✓ YOLOv5 MODEL 1 FEATURE EXTRACTION COMPLETE")
print("=" * 70)

MERGING YOLOv5 CHECKPOINTS

Checkpoint directory:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Checkpoints

Final output:
/content/drive/MyDrive/Image_captioning/Outputs/YOLOv5_Detection/Final/yolov5_detections_full.csv

CHECKPOINT FILES FOUND

Total checkpoint files:
78

checkpoint_00001.csv
  Rows: 2082
  Unique images: 512

checkpoint_00002.csv
  Rows: 1910
  Unique images: 512

checkpoint_00003.csv
  Rows: 1961
  Unique images: 512

checkpoint_00004.csv
  Rows: 2348
  Unique images: 512

checkpoint_00005.csv
  Rows: 1926
  Unique images: 512

checkpoint_00006.csv
  Rows: 2090
  Unique images: 512

checkpoint_00007.csv
  Rows: 2163
  Unique images: 512

checkpoint_00008.csv
  Rows: 2530
  Unique images: 512

checkpoint_00009.csv
  Rows: 2498
  Unique images: 512

checkpoint_00010.csv
  Rows: 2536
  Unique images: 512

checkpoint_00011.csv
  Rows: 2464
  Unique images: 512

checkpoint_00012.csv
  Rows: 2576
  Unique images: 512

checkpoint_00013.csv
  Rows: 2283
 